In [3]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-06 17:56:33--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 
  HTTP/1.1 200 OK
  Connection: keep-alive
  Content-Length: 1115394
  Cache-Control: max-age=300
  Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
  Content-Type: text/plain; charset=utf-8
  ETag: "a82bf4e8979c373a24f616ef6c044f821e18ce64322e5e1280f069f2910b3653"
  Strict-Transport-Security: max-age=31536000
  X-Content-Type-Options: nosniff
  X-Frame-Options: deny
  X-XSS-Protection: 1; mode=block
  X-GitHub-Request-Id: 0E7C:4F49B:2EE28BD:3747732:69D45640
  Accept-Ranges: bytes
  Date: Tue, 07 Apr 2026 00:56:33 GMT
  Via: 1.1 varnish
  X-Served-By: cache-bfi-kbfi7400085-BFI
  X-

In [23]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


### Train and test splits

In [3]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [5]:
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [13]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

### Hyperparameters

In [14]:
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Override for Apple hardware
device = 'mps' if torch.backends.mps.is_available() else device
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

### Types

In [31]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

### Training

In [9]:
model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

0.209729 M parameters
step 0: train loss 4.3491, val loss 4.3521
step 100: train loss 2.6618, val loss 2.6712
step 200: train loss 2.5057, val loss 2.5201
step 300: train loss 2.4361, val loss 2.4422
step 400: train loss 2.3753, val loss 2.3909
step 500: train loss 2.3319, val loss 2.3294
step 600: train loss 2.2695, val loss 2.2850
step 700: train loss 2.2228, val loss 2.2345
step 800: train loss 2.1955, val loss 2.2098
step 900: train loss 2.1499, val loss 2.1738
step 1000: train loss 2.1115, val loss 2.1445
step 1100: train loss 2.0737, val loss 2.1128
step 1200: train loss 2.0468, val loss 2.0949
step 1300: train loss 2.0321, val loss 2.0892
step 1400: train loss 1.9900, val loss 2.0527
step 1500: train loss 1.9758, val loss 2.0542
step 1600: train loss 1.9455, val loss 2.0151
step 1700: train loss 1.9318, val loss 2.0044
step 1800: train loss 1.9172, val loss 2.0080
step 1900: train loss 1.9006, val loss 1.9796
step 2000: train loss 1.8809, val loss 1.9656
step 2100: train loss 1.

### Generate from the model

In [10]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


Oncut be true heir bend; thou come your ffects
His awge. Come, there his duke; I pot gster ats we this come!

KING HENRY VI:
Mois go keak me for our a wake a hand cage in your forge glun
Of will's Whith thou have live I conswer,
God your that our eviesionesth haste:
Gove, hind
to her do Ange then for the amone.
Heres noply a song any.
Beout must hate, that hanst one,
If his us counse sover; as I hadstandied' the strius,
my shore with Takes of the were thy fashul her stand's
Or che More signate. answing, he Contunion:
Sir hatT's platime any contentry.

AYLARDIE:
Wetcus make I thou but shouldil these you, the take kinded Romiolan lord gat I must not imfring this chirst be poisor liender, nem friiends oke caultit with
It basing it brief. Charecion, geave mine:
Have dunking posty to-bot up.

LLAUT:
What will I show! let matter sain:
What, pome stay
the storull tringzer buserve one?

Son my stagest I this grone contratcester that make the vine a swead this drid.

Serving I'll give:
Pase, m

# Questions

## EX1: The n-dimensional tensor mastery challenge: Combine the `Head` and `MultiHeadAttention` into one class that processes all the heads in parallel, treating the heads as another batch dimension (answer is in nanoGPT).

### Types

In [32]:
class FastMultiHeadAttention(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.key = nn.Linear(n_embd, n_embd, bias=False)
        self.query = nn.Linear(n_embd, n_embd, bias=False)
        self.value = nn.Linear(n_embd, n_embd, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):
        B,T,C = x.shape
        head_size = C // self.num_heads
        k = self.key(x)   # (B,T,C)
        k = k.view(B, T, self.num_heads, head_size).transpose(-2, -3) # (B, num_heads, T, head_size)
        q = self.query(x) # (B,T,C)
        q = q.view(B, T, self.num_heads, head_size).transpose(-2, -3) # (B, num_heads, T, head_size)
        wei = q @ k.transpose(-2,-1) * head_size**-0.5 # (B, num_heads, T, head_size) @ (B, num_heads, head_size, T) -> (B, num_heads, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, num_heads, T, T)
        wei = F.softmax(wei, dim=-1) # (B, num_heads, T, T)
        wei = self.dropout(wei) # (B, num_heads, T, T)
        v = self.value(x) # (B,T,C)
        v = v.view(B, T, self.num_heads, head_size).transpose(-2, -3) # (B, num_heads, T, head_size)
        out = wei @ v # (B, num_heads, T, T) @ (B, num_heads, T, head_size) -> (B, num_heads, T, head_size)
        out = out.transpose(-2, -3).reshape(B, T, C)
        out = self.dropout(self.proj(out))
        return out

class FastBlock(Block):
    def __init__(self, n_embd, n_head):
        super().__init__(n_embd, n_head)
        self.sa = FastMultiHeadAttention(n_head)

class FastBigramLanguageModel(BigramLanguageModel):
    def __init__(self):
        super().__init__()
        self.blocks = nn.Sequential(*[FastBlock(n_embd, n_head=n_head) for _ in range(n_layer)])

### Training

In [36]:
model = FastBigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

0.209729 M parameters
step 0: train loss 4.3681, val loss 4.3571
step 100: train loss 2.6536, val loss 2.6663
step 200: train loss 2.4972, val loss 2.5027
step 300: train loss 2.4151, val loss 2.4230
step 400: train loss 2.3332, val loss 2.3366
step 500: train loss 2.2686, val loss 2.2745
step 600: train loss 2.2166, val loss 2.2328
step 700: train loss 2.1795, val loss 2.1994
step 800: train loss 2.1373, val loss 2.1608
step 900: train loss 2.0890, val loss 2.1317
step 1000: train loss 2.0566, val loss 2.0797
step 1100: train loss 2.0147, val loss 2.0861
step 1200: train loss 1.9882, val loss 2.0488
step 1300: train loss 1.9678, val loss 2.0326
step 1400: train loss 1.9444, val loss 2.0323
step 1500: train loss 1.9318, val loss 2.0433
step 1600: train loss 1.8960, val loss 2.0132
step 1700: train loss 1.8987, val loss 2.0205
step 1800: train loss 1.8669, val loss 1.9745
step 1900: train loss 1.8573, val loss 1.9653
step 2000: train loss 1.8371, val loss 1.9515
step 2100: train loss 1.

### Generate from the model

In [37]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


We die is hath any, and use be some was long.

First:
OXle mistif, we reqown; of sours. Too, are the rin is not a furscapios a
casp osiour a loss, horsoss wond reposs
Agicalust of your call absices of certies counds,
And eastingl are Babrery.
Have do's strigge: where have I serbed,
Here I have done s'ull-know this contries:
I thou drancabrook
And is his one hrades may friend God she
hih he child.
The art parsuage as morn a majustanness
He thouse oward He seens.

First Servous,
Hinous things that to decrumia, serdition.
Was lonce, let nature. I had abour a grest in were are and Beear
Not life. Which I plisoubt! old lries.

GLOUCESTER:
Virthart, touch'd of intrue; and so many the put of honourt,
And the give hopp of slizens!

MENAMIO:
My deather oat! and Mail But nee; armshy.

QUEENsES:
B if days of be to sery are Hasteninia;
For proted insciences to her cresolaude to could I not not son,
Whow he giat my wife to drume of Most.

BRUTUS:
We had his tad, Lord
I coutisio? I am have the play

## EX2: Train the GPT on your own dataset of choice! What other data could be fun to blabber on about? (A fun advanced suggestion if you like: train a GPT to do addition of two numbers, i.e. a+b=c. You may find it helpful to predict the digits of c in reverse order, as the typical addition algorithm (that you're hoping it learns) would proceed right to left too. You may want to modify the data loader to simply serve random problems and skip the generation of train.bin, val.bin. You may want to mask out the loss at the input positions of a+b that just specify the problem using y=-1 in the targets (see CrossEntropyLoss ignore_index). Does your Transformer learn to add? Once you have this, swole doge project: build a calculator clone in GPT, for all of +-*/. Not an easy problem. You may need Chain of Thought traces.)

### Building the dataset

In [35]:
stoi = { str(ch):i for i,ch in enumerate(range(0,10)) }
stoi.update({
    '_': 10,
    '=': 11,
    '+': 12,
    '\n': 13,
})
itos = {i:ch for i,ch in enumerate(stoi)}
vocab_size = len(itos.keys())

encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

In [36]:
import random

# data loading
def get_batch():
    max_digits = (block_size - 3) // 3  # block_size = 3*max_digits + 3
    max_num = 10 ** max_digits - 1
    x_problems = []
    y_problems = []
    for _ in range(batch_size):
        a = random.randint(0, max_num)
        b = random.randint(0, max_num)
        c = a + b
        a_str = str(a).zfill(max_digits)
        b_str = str(b).zfill(max_digits)
        c_str = str(c)[::-1].ljust(max_digits + 1, '_')
        s = f"{a_str}+{b_str}={c_str}\n"
        assert len(s) == block_size + 1
        encoded = encode(s)
        x_problems.append(torch.tensor(encoded[:block_size]))

        # Encoding y - for the targets, we want to mask everything up to the first =
        index_after_equals = 2 * max_digits + 1  # = is at this index in y (shifted by 1 from s)
        y_list = [-1] * index_after_equals + encoded[index_after_equals + 1 : block_size + 1]
        y_problems.append(torch.tensor(y_list))
    x = torch.stack(x_problems)
    y = torch.stack(y_problems)

    x, y = x.to(device), y.to(device)
    return x, y
    
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch()
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

### Types

In [ ]:
class AdditionLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[FastBlock(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            # This is the only difference between this and the bigram model - ignoring the character class -1, which is everything up to the first = in the target logits
            loss = F.cross_entropy(logits, targets, ignore_index=-1)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

### Hyperparameters

In [58]:
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 12 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Override for Apple hardware
device = 'mps' if torch.backends.mps.is_available() else device
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

### Training

In [59]:
model = AdditionLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch()

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

0.20187 M parameters
step 0: train loss 2.9731, val loss 2.9692
step 100: train loss 1.4198, val loss 1.4177
step 200: train loss 1.3952, val loss 1.3944
step 300: train loss 1.3958, val loss 1.3939
step 400: train loss 1.3922, val loss 1.3907
step 500: train loss 1.3679, val loss 1.3653
step 600: train loss 1.2526, val loss 1.2481
step 700: train loss 1.0829, val loss 1.0847
step 800: train loss 0.7371, val loss 0.7422
step 900: train loss 0.4557, val loss 0.4569
step 1000: train loss 0.2266, val loss 0.2283
step 1100: train loss 0.1119, val loss 0.1111
step 1200: train loss 0.0608, val loss 0.0603
step 1300: train loss 0.0989, val loss 0.0977
step 1400: train loss 0.0225, val loss 0.0242
step 1500: train loss 0.0065, val loss 0.0064
step 1600: train loss 0.0046, val loss 0.0045
step 1700: train loss 0.0039, val loss 0.0041
step 1800: train loss 0.0035, val loss 0.0031
step 1900: train loss 0.0026, val loss 0.0026
step 2000: train loss 0.0022, val loss 0.0022
step 2100: train loss 0.0

### Sampling from the model

In [61]:
for i in range(0, 10):
    # TODO: This feels kind of janky, we should clean up the token generation
    max_digits = (block_size - 3) // 3
    a, b = random.randint(0, 999), random.randint(0, 999)
    prompt = f"{a:0{max_digits}}+{b:0{max_digits}}="
    encoded_problem = torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    print(decode(m.generate(encoded_problem, max_new_tokens=max_digits + 1)[0].tolist()))

132+143=572_
777+656=3341
149+003=251_
797+410=7021
828+237=5601
308+865=3711
300+486=687_
463+929=2931
044+520=465_
391+019=014_


## EX3: Find a dataset that is very large, so large that you can't see a gap between train and val loss. Pretrain the transformer on this data, then initialize with that model and finetune it on tiny shakespeare with a smaller number of steps and lower learning rate. Can you obtain a lower validation loss by the use of pretraining?

### Download the dataset

In [2]:
import asyncio
import aiohttp
import aiofiles
from huggingface_hub import hf_hub_download
import numpy as np

In [5]:
files_list_path = hf_hub_download("deepmind/pg19", "data/train_files.txt", repo_type="dataset")
with open(files_list_path) as f:
    train_files = sorted(f.read().splitlines())
print(f"Found {len(train_files)} books to download")

Found 28602 books to download


In [ ]:
async def fetch_book(session, semaphore, path):
    url = f"https://storage.googleapis.com/deepmind-gutenberg/{path}"
    async with semaphore:
        try:
            async with session.get(url) as resp:
                resp.raise_for_status()
                return await resp.text(encoding="utf-8")
        except Exception as e:
            print(f"  Warning: skipped {path} ({e})")
            return None

async def download_pg19(train_files, out_path="pg19_train.txt", batch_size=256):
    semaphore = asyncio.Semaphore(64)
    written = 0

    async with aiohttp.ClientSession() as session:
        async with aiofiles.open(out_path, "w", encoding="utf-8") as out:
            for batch_start in range(0, len(train_files), batch_size):
                batch = train_files[batch_start : batch_start + batch_size]
                texts = await asyncio.gather(
                    *[fetch_book(session, semaphore, path) for path in batch]
                )
                for text in texts:
                    if text is not None:
                        await out.write(text)
                        await out.write("\n\n")
                        written += 1
                print(f"  {batch_start + len(batch)}/{len(train_files)} fetched, {written} written")

    print(f"Done. {written} books saved to {out_path}")

await download_pg19(train_files)

  256/28602 fetched, 256 written
  512/28602 fetched, 512 written
  768/28602 fetched, 768 written
  1024/28602 fetched, 1024 written
  1280/28602 fetched, 1280 written
  1536/28602 fetched, 1536 written
  1792/28602 fetched, 1792 written
  2048/28602 fetched, 2048 written
  2304/28602 fetched, 2304 written
  2560/28602 fetched, 2560 written
  2816/28602 fetched, 2816 written
  3072/28602 fetched, 3072 written
  3328/28602 fetched, 3328 written
  3584/28602 fetched, 3584 written
  3840/28602 fetched, 3840 written
  4096/28602 fetched, 4096 written
  4352/28602 fetched, 4352 written
  4608/28602 fetched, 4608 written
  4864/28602 fetched, 4864 written
  5120/28602 fetched, 5120 written
  5376/28602 fetched, 5376 written
  5632/28602 fetched, 5632 written
  5888/28602 fetched, 5888 written
  6144/28602 fetched, 6144 written
  6400/28602 fetched, 6400 written
  6656/28602 fetched, 6656 written
  6912/28602 fetched, 6912 written
  7168/28602 fetched, 7168 written
  7424/28602 fetched, 7424

In [3]:
chars = set()
with open("pg19_train.txt", "r", encoding="utf-8") as f:
    while chunk := f.read(1_000_000):  # 1MB at a time
        chars.update(chunk)
chars = sorted(chars)
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

 	





 !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}~
 ¡¢£¤¥¦§¨©ª«¬­®¯°±²³´µ¶·¸¹º»¼½¾¿ÀÁÂÃÄÅÆÇÈÉÊËÌÍÎÏÐÑÒÓÔÕÖ×ØÙÚÛÜÝÞßàáâãäåæçèéêëìíîïðñòóôõö÷øùúûüýþÿĀāĂăąĆćĈĉċČčďđĒēĔĕĖėĘęĚěĜĝğĠġĤĦħĨĩĪīĬĭįİıĵļŁłŃńŇňŉŊŋŌōŎŏőŒœŔŕŘřŚśŜŝŞşŠšţťŧŨũŪūŬŭŮůűųŴŵŶŷŸźŻżŽžſƀƆƎƐƒƕƖƚƥƧưƵƷƿǁǎǏǐǒǔǕǖǝǟǡǢǣǥǧǪǫǭǴǵǷǹǽȗȚțȜȝȣȦȧȩȫȯȱȲȳȻȼɅɑɒɔəɛɣɥɧɩɪɫɭɯɱɲɳɶɹʃʅʇʊʌʍʒʘʞʠʰʲʳʷʸʹʺʻʼʽʾʿˀˆˈˉˌːˑ˔˕˘˙˚˛˜˝ˡˢˣ˭˳̴̵̶̷̸̢̧̨̝̠̣̤̥̦̩̭̯̱̲͕̀́̂̃̄̅̆̇̈̉̊̋̌̍̑̓̾́͂̓̈́͆͐͒ͣͤͥͦͧͨͩͪͬͭͮ͜͝͞͠͡ͅͰʹ͵;΄΅Ά·ΈΉΊΌΎΏΐΑΒΓΔΕΖΗΘΙΚΛΜΝΞΟΠΡΣΤΥΦΧΨΩΪάέήίΰαβγδεζηθικλμνξοπρςστυφχψωϊϋόύώϐϑϒϕϖϗϘϙϚϛϜϝϞϟϡϥϫϯϰϱϲϴϵ϶ϹϺϻϽЂЄІЈЉЊЏАБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯабвгдежзийклмнопрстуфхцчшщъыьэюяёђєіјћўѢѣѲѳѴѵҗҫӑӱԀԂԊԱԵԶԸԻՆՓաբգեզթիլխծկհղմնոչպսվտրցւփքօևְֱֲֳִֵֶַָֹֺֻּֽ֑֖֛֢֣֤֥֦֧֚֭֮֒֓֔֕֗֘֙֜֝֞֟֠֡֨֩֫֬֯־׀ׁׂ׃ׇׄאבגדהוזחטיךכלםמןנסעףפץצקרשתױײ׳״،؟ءآأؤإئابةتثجحخدذرزسشصضطظعغفقكلمنهوىيًٌَُِّْ٘٢٥٦٧ٱپچڙکگۃیܐܒܓܕܗܘܙܚܛܝܠܡܢܣܤܥܦܩܪܫܬܰܳܵܿࣨटठनबमरलवसािी्ढ़ന་ཁགངཆཇཉཎདནཔཕབམཚཛའརལསཧཨཱིེོུཾྃྐྒྟྨྭྱྲྷჄᎻᏙᐯᐱᒥᒪᕼᖤᗜᚠᚢᚤᚦᚨᚩᚭᚮᚱᚳᚴᚷᚺᚻᚼᚾᚿᛁᛂᛅᛆᛉᛋᛌᛍ

In [4]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[93, 94, 94, 21, 105, 93, 90, 103, 90]
hii there


In [5]:
with open("pg19_train.txt", "r", encoding="utf-8") as f, open("pg19_train.bin", "wb") as out:
    while chunk := f.read(1_000_000):
        ids = np.array(encode(chunk), dtype=np.uint16)
        ids.tofile(out)

In [17]:
data = np.memmap("pg19_train.bin", dtype=np.uint16, mode="r")
n = int(0.9 * len(data))
train_data = data[:n]   # still a memmap, no data copied
val_data   = data[n:] 
print(f"Total tokens: {len(data):,}")

Total tokens: 11,425,133,528


### Train and test splits

In [18]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i  :i+block_size  ].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+block_size+1].astype(np.int64)) for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

### Pretraining

In [19]:
model = FastBigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

0.653618 M parameters
step 0: train loss 8.2945, val loss 8.2971
step 100: train loss 2.9243, val loss 2.8900
step 200: train loss 2.7284, val loss 2.6760
step 300: train loss 2.6157, val loss 2.5721
step 400: train loss 2.4980, val loss 2.4751
step 500: train loss 2.4277, val loss 2.3999
step 600: train loss 2.3690, val loss 2.3530
step 700: train loss 2.3151, val loss 2.2978
step 800: train loss 2.2658, val loss 2.2608
step 900: train loss 2.2441, val loss 2.2235
step 1000: train loss 2.2032, val loss 2.1894
step 1100: train loss 2.1762, val loss 2.1574
step 1200: train loss 2.1489, val loss 2.1417
step 1300: train loss 2.1344, val loss 2.1183
step 1400: train loss 2.1179, val loss 2.0951
step 1500: train loss 2.0924, val loss 2.0891
step 1600: train loss 2.0765, val loss 2.0602
step 1700: train loss 2.0796, val loss 2.0569
step 1800: train loss 2.0590, val loss 2.0400
step 1900: train loss 2.0468, val loss 2.0182
step 2000: train loss 2.0267, val loss 2.0200
step 2100: train loss 2.

### Sampling after pretraining

In [21]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

 ; there led at the
provend of the
must of poenited could your bats, in lannagy or a try of pistlened all cover the stect that have been
are boldifulls Is have as never ead when it, maned, which allow singurates,
whether fish is subjects all
yeames, and that Rollamlake Co them wolln Evensides if the pholings, who shindy he he seent bort. No. Ill it the Stiotol are has like
and here it when sensilish
fumpn brished to stift this agistor"
  Told omles, an who not-the lettlenow, and sings; one, negt 


### Shakespeare fluff for finetuning

In [23]:
# Reload Shakespeare text
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Re-encode using the pg19 vocab (stoi/itos unchanged)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [26]:
# Redefine this to accommodate data as a tensor
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

### Finetuning

In [27]:
# Tweak iterations and learning rate
max_iters = 2000
learning_rate = 1e-4

In [28]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [29]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 2.1005, val loss 2.1903
step 100: train loss 1.9276, val loss 1.9760
step 200: train loss 1.8898, val loss 1.9347
step 300: train loss 1.8496, val loss 1.8993
step 400: train loss 1.8467, val loss 1.8820
step 500: train loss 1.8321, val loss 1.8650
step 600: train loss 1.8159, val loss 1.8566
step 700: train loss 1.8113, val loss 1.8580
step 800: train loss 1.8015, val loss 1.8459
step 900: train loss 1.7983, val loss 1.8404
step 1000: train loss 1.7820, val loss 1.8458
step 1100: train loss 1.7823, val loss 1.8319
step 1200: train loss 1.7674, val loss 1.8299
step 1300: train loss 1.7716, val loss 1.8215
step 1400: train loss 1.7739, val loss 1.8250
step 1500: train loss 1.7538, val loss 1.8205
step 1600: train loss 1.7436, val loss 1.8183
step 1700: train loss 1.7508, val loss 1.8153
step 1800: train loss 1.7465, val loss 1.8107
step 1900: train loss 1.7412, val loss 1.8092
step 1999: train loss 1.7330, val loss 1.8087


### Sampling after finetuning

In [30]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

 s tell upon erronds
And the cruys made very harts hears,
I was not go and neor those despect
Hardst do bown'd: she she famines:
But the proven. I to mine our head may mertaint
she row yet offsely ago! so there will enter on thee,
To they powing; work off-genenerthod to yours:
For she my intended's son.
What, I am plaso, men.

CANETE:
And foll aken
The givess breather mother.

LEUNIO:
What ill, our and with partized in men deece, and it mest.
Thereth are-than this brothing one my six though adfark still so stiil withing, say 'i, when perster you,
 he kirst for Dollesss of Ruch would: with the bleme!

CORBAINCUKE:
That Coun, a not us love was thou
thoster, been as they Jught, pistial whise for between
Shall tisting did our lown.
That, with ammes! she me than my panysoly.

JY SFWIIBIIM:
Wheth a harge what neved her, ton't your jirtigh us.

MAUTER'S:
In'y he're on: do the histing diver from:
Mercoust, and mine mildishersful beater of Yi‐
To the fitt such, not light thy well.
Last lears, h